## TF-IDF Retrieval Experiment

This experiment evaluates TF-IDF as a lexical retrieval method for mapping
cyber threat intelligence sentences to MITRE ATT&CK techniques.

The MITRE Enterprise ATT&CK technique names and descriptions form the
retrieval corpus. Single-label TRAM threat intelligence sentences are used
as queries, with their associated ATT&CK technique IDs serving as
ground-truth labels.

For each query, TF-IDF and cosine similarity are used to rank all MITRE
ATT&CK techniques.


**Pipeline of this experiement**

1. tram dataset sentence
2. TF-IDF
3. Cosine Similarity
4. Rank all MITRE Techniques
5. where did the correct technique show up?

**Notes**

stemming/lemmatization was not performed in this experiement since this is only establishing a baseline. If we start heavily optimizing TF-IDF before establishing the baseline, it becomes harder to tell what caused improvements. After researching, I also determined that some cybersecurity terms are unique and should not be shortened or revert to their root forms.

**evaluation for this experiment**

- used recall at 1
- recall @5
- recall@10
- MRR

In [1]:
import pandas as pd

mitre_df = pd.read_csv(
    "mitre_techniques.csv"
)

tram_df = pd.read_csv(
    "tram_single_label_compatible.csv"
)

In [2]:
#check
print("MITRE corpus:", mitre_df.shape)
print("TRAM queries:", tram_df.shape)

print(
    "MITRE techniques:",
    mitre_df["technique_id"].nunique()
)

print(
    "TRAM ground-truth techniques:",
    tram_df["technique_id"].nunique()
)

MITRE corpus: (697, 7)
TRAM queries: (3077, 5)
MITRE techniques: 697
TRAM ground-truth techniques: 47


In [3]:
mitre_df.head()

,technique_id,name,description,is_subtechnique,platforms,tactics,stix_id
0,T1055.011,Extra Window Memory Injection,Adversaries may inject malicious code into pro...,True,['Windows'],"['stealth', 'privilege-escalation']",attack-pattern--0042a9f5-f053-4769-b3ef-9ad018...
1,T1053.005,Scheduled Task,Adversaries may abuse the Windows Task Schedul...,True,['Windows'],"['execution', 'persistence', 'privilege-escala...",attack-pattern--005a06c6-14bf-4118-afa0-ebcd8a...
2,T1205.002,Socket Filters,Adversaries may attach filters to a network so...,True,"['Linux', 'macOS', 'Windows']","['stealth', 'persistence', 'command-and-control']",attack-pattern--005cc321-08ce-4d17-b1ea-cb5275...
3,T1560.001,Archive via Utility,Adversaries may use utilities to compress and/...,True,"['Linux', 'macOS', 'Windows']",['collection'],attack-pattern--00f90846-cbd1-4fc5-9233-df5c2b...
4,T1021.005,VNC,Adversaries may use [Valid Accounts](https://a...,True,"['Linux', 'Windows', 'macOS']",['lateral-movement'],attack-pattern--01327cde-66c4-4123-bf34-5f258d...


In [4]:
tram_df.head()

,sentence,technique_id,doc_title,split,technique_name
0,It spreads to Microsoft Windows machines using...,T1210,NotPetya Technical Analysis A Triple Threat F...,train,Exploitation of Remote Services
1,"Once a machine is infected by NotPetya, a seri...",T1570,NotPetya Technical Analysis A Triple Threat F...,train,Lateral Tool Transfer
2,Ransomware DLL C:\windows\perfc.dat,T1140,NotPetya Technical Analysis A Triple Threat F...,train,Deobfuscate/Decode Files or Information
3,The malware decompresses its resource named 0x...,T1140,NotPetya Technical Analysis A Triple Threat F...,train,Deobfuscate/Decode Files or Information
4,The malware is a DLL that is launched using ru...,T1218.011,NotPetya Technical Analysis A Triple Threat F...,train,Rundll32


In [6]:
#MITRE dataset NAME + DESCRIPTION

mitre_df["retrieval_text"] = (
    mitre_df["name"].fillna("")
    + ". "
    + mitre_df["description"].fillna("")
)

**combine technique name and description into searchable text**

In [7]:
print(mitre_df.loc[0, "retrieval_text"])

Extra Window Memory Injection. Adversaries may inject malicious code into process via Extra Window Memory (EWM) in order to evade process-based defenses as well as possibly elevate privileges. EWM injection is a method of executing arbitrary code in the address space of a separate live process. 

Before creating a window, graphical Windows-based processes must prescribe to or register a windows class, which stipulate appearance and behavior (via windows procedures, which are functions that handle input/output of data).(Citation: Microsoft Window Classes) Registration of new windows classes can include a request for up to 40 bytes of EWM to be appended to the allocated memory of each instance of that class. This EWM is intended to store data specific to that window and has specific application programming interface (API) functions to set and get its value. (Citation: Microsoft GetWindowLong function) (Citation: Microsoft SetWindowLong function)

Although small, the EWM is large enough t

**CLEAN TEXT**

**NOTES**

- MITRE data has citations and author names. removing them, not very useful for mapping the cyberattacks

In [8]:
import re

def clean_mitre_text(text):
    text = str(text)

    # Remove MITRE citation markers
    text = re.sub(
        r"\(Citation:[^)]+\)",
        "",
        text
    )

    # Convert Markdown links to visible text
    text = re.sub(
        r"\[([^\]]+)\]\([^)]+\)",
        r"\1",
        text
    )

    # Remove backticks but preserve contents
    text = text.replace("`", "")

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [9]:
mitre_df["retrieval_text"] = (
    mitre_df["retrieval_text"]
    .apply(clean_mitre_text)
)

**CLEAN TRAM DATASET**

In [10]:
tram_df["query_text"] = (
    tram_df["sentence"]
    .fillna("")
    .str.strip()
)

**FIT TF-IDF**

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english"
)

In [13]:
mitre_matrix = vectorizer.fit_transform(
    mitre_df["retrieval_text"]
)

In [14]:
print("TF-IDF matrix shape:", mitre_matrix.shape)
print("Vocabulary size:", len(vectorizer.vocabulary_))

TF-IDF matrix shape: (697, 5907)
Vocabulary size: 5907


In [15]:
#tram df
query_matrix = vectorizer.transform(
    tram_df["query_text"]
)

In [16]:
print(
    "Query matrix shape:",
    query_matrix.shape
)

Query matrix shape: (3077, 5907)


**Attempt 1 query before proceeding**

In [17]:
query_index = 0

query_text = tram_df.iloc[query_index]["query_text"]
ground_truth = tram_df.iloc[query_index]["technique_id"]

print("QUERY:")
print(query_text)

print("\nGROUND TRUTH:")
print(ground_truth)

QUERY:
It spreads to Microsoft Windows machines using several propagation methods, including the EternalBlue exploit for the CVE-2017-0144 vulnerability in the SMB service.

GROUND TRUTH:
T1210


In [19]:
from sklearn.metrics.pairwise import cosine_similarity
scores = cosine_similarity(
    query_matrix[query_index],
    mitre_matrix
).flatten()

In [21]:
import numpy as np

#rank the scores
top_indices = np.argsort(scores)[::-1][:10]

In [22]:
results = mitre_df.iloc[top_indices][
    ["technique_id", "name"]
].copy()

results["score"] = scores[top_indices]

results

,technique_id,name,score
126,T1588.006,Vulnerabilities,0.269865
233,T1021.002,SMB/Windows Admin Shares,0.234434
502,T1587.004,Exploits,0.180971
252,T1595.002,Vulnerability Scanning,0.171959
121,T1543.003,Windows Service,0.170523
664,T1588.005,Exploits,0.168145
196,T1190,Exploit Public-Facing Application,0.158155
511,T1203,Exploitation for Client Execution,0.142210
591,T1651,Cloud Administration Command,0.129537
427,T1210,Exploitation of Remote Services,0.127446


In [23]:
#summarzing this first attempt
print("Query:")
print(query_text)

print("\nCorrect technique:")
print(ground_truth)

print("\nTop 10 retrieved techniques:")

for rank, idx in enumerate(top_indices, start=1):
    print(
        f"{rank}. "
        f"{mitre_df.iloc[idx]['technique_id']} - "
        f"{mitre_df.iloc[idx]['name']} "
        f"(score={scores[idx]:.4f})"
    )

Query:
It spreads to Microsoft Windows machines using several propagation methods, including the EternalBlue exploit for the CVE-2017-0144 vulnerability in the SMB service.

Correct technique:
T1210

Top 10 retrieved techniques:
1. T1588.006 - Vulnerabilities (score=0.2699)
2. T1021.002 - SMB/Windows Admin Shares (score=0.2344)
3. T1587.004 - Exploits (score=0.1810)
4. T1595.002 - Vulnerability Scanning (score=0.1720)
5. T1543.003 - Windows Service (score=0.1705)
6. T1588.005 - Exploits (score=0.1681)
7. T1190 - Exploit Public-Facing Application (score=0.1582)
8. T1203 - Exploitation for Client Execution (score=0.1422)
9. T1651 - Cloud Administration Command (score=0.1295)
10. T1210 - Exploitation of Remote Services (score=0.1274)


**moving onto retrieving all queries using tfidf**

In [24]:
id_to_index = {
    technique_id: idx
    for idx, technique_id
    in enumerate(mitre_df["technique_id"])
}

In [25]:
#calcualte similarity
similarity_matrix = cosine_similarity(
    query_matrix,
    mitre_matrix
)

In [27]:
ranks = []

for i in range(len(tram_df)):

    ground_truth = tram_df.iloc[i]["technique_id"]

    correct_index = id_to_index[ground_truth]

    scores = similarity_matrix[i]

    ranked_indices = np.argsort(scores)[::-1]

    correct_rank = (
        np.where(
            ranked_indices == correct_index
        )[0][0]
        + 1
    )

    ranks.append(correct_rank)

In [28]:
tram_df["tfidf_rank"] = ranks

In [29]:
tram_df[
    [
        "sentence",
        "technique_id",
        "tfidf_rank"
    ]
].head(10)

,sentence,technique_id,tfidf_rank
0,It spreads to Microsoft Windows machines using...,T1210,10
1,"Once a machine is infected by NotPetya, a seri...",T1570,31
2,Ransomware DLL C:\windows\perfc.dat,T1140,122
3,The malware decompresses its resource named 0x...,T1140,22
4,The malware is a DLL that is launched using ru...,T1218.011,1
5,It is executed with the following arguments: #...,T1057,202
6,The credential theft module is dropped if the ...,T1106,89
7,The malware decompresses a resource (0x1 if th...,T1140,28
8,It calls OpenProcess on lsass.exe with access ...,T1003.001,1
9,"Server service, which both enforces security p...",T1003.001,2


**moving on to evaluation using recall**

In [30]:
#recall@1
recall_at_1 = (
    tram_df["tfidf_rank"] <= 1
).mean()

#recall@5
recall_at_5 = (
    tram_df["tfidf_rank"] <= 5
).mean()

#recall@10
recall_at_10 = (
    tram_df["tfidf_rank"] <= 10
).mean()

In [31]:
print(f"Recall@1:  {recall_at_1:.3f}")
print(f"Recall@5:  {recall_at_5:.3f}")
print(f"Recall@10: {recall_at_10:.3f}")

Recall@1:  0.129
Recall@5:  0.305
Recall@10: 0.399


**notes for eval**

- 12.9% of queries have the correct answer at #1.
- 30.5% have it somewhere in the first 5.
- 39.9% have it somewhere in the first 10.

In [32]:
mrr = (
    1 / tram_df["tfidf_rank"]
).mean()

print(f"MRR: {mrr:.3f}")

MRR: 0.221


**EXplanation**

- MRR ranges from 0 (correct answer rank very low)  to 1 (correct answer rank first)
- Result shows a lower score meaning correct answer shows but at a lower rank

In [33]:
#summary table
tfidf_results = pd.DataFrame({
    "method": ["TF-IDF"],
    "recall@1": [recall_at_1],
    "recall@5": [recall_at_5],
    "recall@10": [recall_at_10],
    "mrr": [mrr]
})

tfidf_results

,method,recall@1,recall@5,recall@10,mrr
0,TF-IDF,0.129347,0.304517,0.399415,0.220974


In [34]:
#save rank for later error analysis
tfidf_query_results = tram_df[
    [
        "sentence",
        "technique_id",
        "tfidf_rank",
        "split"
    ]
].copy()

tfidf_query_results.to_csv(
    "tfidf_query_results.csv",
    index=False
)

tfidf_results.to_csv(
    "tfidf_summary.csv",
    index=False
)